# 🏦 Credit Risk Model — Loan Default Prediction
Predict loan defaults using Logistic Regression and XGBoost on LendingClub-style data.

## 1. Imports

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from generate_data import generate_lending_club_data
from eda import (
    plot_target_distribution,
    plot_numeric_distributions,
    plot_correlation_matrix,
    plot_categorical_default_rates
)
from model import run_pipeline
from roc_auc import plot_roc_curve, get_feature_importance, plot_confusion_matrix, get_classification_report

## 2. Data Loading

In [ ]:
DATA_PATH = os.path.join('data', 'lending_club.csv')
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f'Loaded existing dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
else:
    print('Dataset not found — generating synthetic LendingClub-style data...')
    os.makedirs('data', exist_ok=True)
    df = generate_lending_club_data(n_samples=10000, random_state=42)
    df.to_csv(DATA_PATH, index=False)
    print(f'Generated and saved dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head()

## 3. Exploratory Data Analysis

### 3.1 Target Distribution

In [ ]:
fig = plot_target_distribution(df)
fig.show()

### 3.2 Numeric Feature Distributions

In [ ]:
fig = plot_numeric_distributions(df)
fig.show()

### 3.3 Correlation Matrix

In [ ]:
fig = plot_correlation_matrix(df)
fig.show()

### 3.4 Categorical Feature Default Rates

In [ ]:
fig = plot_categorical_default_rates(df)
fig.show()

## 4. Preprocessing & Train/Test Split

In [ ]:
results = run_pipeline(df)
lr_model      = results['lr_model']
xgb_model     = results['xgb_model']
X_test        = results['X_test']
y_test        = results['y_test']
lr_probs      = results['lr_probs']
xgb_probs     = results['xgb_probs']
lr_preds      = results['lr_preds']
xgb_preds     = results['xgb_preds']
feature_names = results['feature_names']
metrics       = results['metrics']
print('Pipeline complete.')
print(f"Train size : {results['train_size']:,}")
print(f"Test size  : {results['test_size']:,}")

## 5. Model Training Results

In [ ]:
print('=== Logistic Regression ===')
for k, v in metrics['logistic_regression'].items():
    print(f'  {k:<12}: {v:.4f}')
print()
print('=== XGBoost ===')
for k, v in metrics['xgboost'].items():
    print(f'  {k:<12}: {v:.4f}')

## 6. ROC Curve

In [ ]:
fig = plot_roc_curve(
    y_test,
    lr_probs,
    xgb_probs,
    lr_auc=metrics['logistic_regression']['roc_auc'],
    xgb_auc=metrics['xgboost']['roc_auc']
)
fig.show()

## 7. Feature Importance

In [ ]:
lr_importance = get_feature_importance(lr_model, feature_names, model_type='lr')
xgb_importance = get_feature_importance(xgb_model, feature_names, model_type='xgb')
fig_lr = px.bar(
    lr_importance.head(15),
    x='importance',
    y='feature',
    orientation='h',
    title='Logistic Regression — Top 15 Feature Importances (|Coefficient|)',
    color='importance',
    color_continuous_scale='Blues'
)
fig_lr.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_lr.show()
fig_xgb = px.bar(
    xgb_importance.head(15),
    x='importance',
    y='feature',
    orientation='h',
    title='XGBoost — Top 15 Feature Importances (Gain)',
    color='importance',
    color_continuous_scale='Oranges'
)
fig_xgb.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_xgb.show()

## 8. Confusion Matrices

In [ ]:
fig_lr_cm = plot_confusion_matrix(y_test, lr_preds, title='Logistic Regression — Confusion Matrix')
fig_lr_cm.show()
fig_xgb_cm = plot_confusion_matrix(y_test, xgb_preds, title='XGBoost — Confusion Matrix')
fig_xgb_cm.show()

## 9. Classification Reports

In [ ]:
lr_report = get_classification_report(y_test, lr_preds)
print('Logistic Regression — Classification Report')
display(pd.DataFrame(lr_report).transpose().round(4))
xgb_report = get_classification_report(y_test, xgb_preds)
print('XGBoost — Classification Report')
display(pd.DataFrame(xgb_report).transpose().round(4))

## 10. Save Results

In [ ]:
os.makedirs('data', exist_ok=True)
output_path = os.path.join('data', 'metrics.json')
with open(output_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved to {output_path}')
print(json.dumps(metrics, indent=2))